In [5]:
import os
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [6]:

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

### Connect to Chroma, use Hugging Face all-MiniLM-L6-v2 

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9093.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
retriever = vectorstore.as_retriever()
# llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)

In [9]:
retriever.invoke("Who is Avery?")

[Document(id='bc55faeb-5d0f-43a9-9d56-0f9bec71b45b', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [12]:
llm.invoke("Who is Avery?")

AIMessage(content='"Avery" can refer to several different things, depending on the context. Here are the most common interpretations:\n\n1.  **A Common Given Name:** Avery is a popular given name, used for both males and females. If you\'re asking about a specific person named Avery, I would need more context (e.g., "Avery from the show...", "My friend Avery...", "Avery, the author...").\n\n2.  **Avery Dennison (Company):** This is a well-known global manufacturing company specializing in various products, including:\n    *   **Adhesive materials:** For labels, graphics, and industrial applications.\n    *   **Labels and tags:** For apparel, retail, and supply chain management.\n    *   **Office products:** They are particularly famous for their line of printable labels (address labels, filing labels, etc.), binders, dividers, and other office supplies.\n\n3.  **A Character:** Avery could be a character in a book, movie, TV show, or video game.\n\n4.  **A Place:** There are towns, stre

In [13]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [14]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [15]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster (note the spelling) is the Co-Founder and Chief Executive Officer (CEO) of Insurellm.\n\nShe co-founded Insurellm in 2015 and has been instrumental in guiding the company to its position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise. Before launching Insurellm, she was a Senior Product Manager at Innovate Insurance Solutions.'

In [ ]:
gr.ChatInterface(answer_question).launch()